# 1 - Data Exploration

Explore the Oxford Flowers 102 dataset: class distribution, image sizes, and sample visualizations.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import Config
from src.utils import set_global_seed, load_oxford_flowers_metadata

cfg = Config()
set_global_seed(cfg.seed)

labels, train_ids, valid_ids, test_ids = load_oxford_flowers_metadata(cfg.dataset_dir)

print(f'Total images: {len(labels)}')
print(f'Classes: {cfg.num_classes}')
print(f'Train: {len(train_ids)} | Val: {len(valid_ids)} | Test: {len(test_ids)}')

## Class Distribution

In [ ]:
train_labels = labels[train_ids - 1]
val_labels = labels[valid_ids - 1]
test_labels = labels[test_ids - 1]

distribution = pd.DataFrame({
    'class_id': range(cfg.num_classes),
    'train': [sum(train_labels == i) for i in range(cfg.num_classes)],
    'val': [sum(val_labels == i) for i in range(cfg.num_classes)],
    'test': [sum(test_labels == i) for i in range(cfg.num_classes)],
})
distribution['total'] = distribution[['train', 'val', 'test']].sum(axis=1)

print(f"Min images per class: {distribution['total'].min()}")
print(f"Max images per class: {distribution['total'].max()}")
print(f"Avg images per class: {distribution['total'].mean():.1f}")

plt.figure(figsize=(18, 5))
plt.bar(distribution['class_id'], distribution['total'])
plt.title('Images per Class - Oxford Flowers 102')
plt.xlabel('Class ID')
plt.ylabel('Count')
plt.xticks(range(0, 102, 5))
plt.grid(axis='y', alpha=0.3)
plt.show()

## Sample Images

In [ ]:
import random
from PIL import Image

random.seed(cfg.seed)
sample_ids = random.sample(list(train_ids), 6)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, img_id in zip(axes.flat, sample_ids):
    img = Image.open(cfg.images_dir / f'image_{img_id:05d}.jpg').convert('RGB')
    ax.imshow(img)
    ax.set_title(f'ID: {img_id} | Class: {labels[img_id - 1]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Image Dimensions

In [ ]:
from collections import Counter

sizes = Counter()
for img_path in cfg.images_dir.glob('*.jpg'):
    with Image.open(img_path) as img:
        sizes[img.size] += 1

print(f'Unique image sizes: {len(sizes)}')
print('\nMost common dimensions:')
for size, count in sizes.most_common(10):
    print(f'  {size}: {count} images')